# 🔧 Healthcare Data Solutions - Deployment Configuration

**Central configuration notebook for all HDS deployment operations**

This notebook provides shared configuration that can be imported by all deployment notebooks using:

```python
%run /path/to/common_deployment_config
```

---

## 📌 What This Provides

1. **Artifact Configuration** - Version, container, source paths
2. **Workspace Context** - Auto-detected workspace ID, endpoint URI
3. **Environment Settings** - Target environment name
4. **Lakehouse Mappings** - Logical to physical lakehouse name mappings
5. **Solution Metadata** - Solution name, prefixes, admin database
6. **Helper Functions** - Common utilities for resolution
7. **Path Construction** - Pre-built paths for artifacts, tables, libraries

---

## 🎯 Usage in Other Notebooks

```python
# At the top of your deployment notebook:
%run common_deployment_config

# Then use the configuration:
print(f"Deploying version: {ARTIFACT_VERSION}")
print(f"Workspace: {WORKSPACE_ID}")
print(f"Base dist path: {BASE_DIST_PATH}")
```

---

> **Note:** Update only the **"User Configuration"** section below. All other values are auto-detected or derived.


In [ ]:
# =============================================================================
# COMMON IMPORTS - Used across all deployment notebooks
# =============================================================================

from notebookutils import mssparkutils
import json
import re
import base64
from typing import Dict, List, Optional, Any, Tuple
from sempy import fabric
from sempy.fabric import FabricRestClient

print("✓ Common imports loaded")


## 1️⃣ User Configuration

**⚠️ EDIT THESE VALUES FOR YOUR DEPLOYMENT**

> **Note:** Workspace name is now auto-detected from the current workspace - no manual configuration needed!

In [ ]:
# =============================================================================
# USER CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# Artifact version (semantic versioning: major.minor.patch)
ARTIFACT_VERSION = "1.4.0"

# OneLake container and lakehouse hosting the distribution artifacts
ARTIFACT_LAKEHOUSE_NAME = "deployment_lakehouse"

# Library package patterns to discover (for environment deployment)
LIBRARY_PACKAGE_PATTERNS = ["hds", "dtt"]


# Target Fabric Environment name
# - Leave empty ("") to auto-select if only 1 environment exists
# - Provide full or partial name for matching (case-insensitive)
TARGET_ENVIRONMENT_NAME = "environment"

# Administration database/lakehouse BASE name (must match manifest key)
# ⚠️ IMPORTANT: Use the exact key from LakehouseHydrationManifest.json
# With COMPANY_PREFIX="healthcare1" and TECHNICAL_PREFIX="msft", becomes: "healthcare1_msft_admin"
# With COMPANY_PREFIX="" and TECHNICAL_PREFIX="msft", becomes: "msft_admin"
ADMIN_DB_NAME = "admin"

# Solution/workspace identifier (BASE name - prefixes applied when used)
# Typically set to the same value as ADMIN_DB_NAME since the solution metadata
# is stored in the admin lakehouse. During deployment, %%solution_name%% placeholder
# will be replaced with the admin lakehouse ID.
SOLUTION_NAME = "admin"

# Artifact naming prefixes (optional - leave empty to skip)
# Pattern: {COMPANY_PREFIX}_{TECHNICAL_PREFIX}_{artifact_name}
# Examples (assuming TECHNICAL_PREFIX="tech" for clarity):
#   - Both set: "contoso_tech_my_environment"
#   - Only technical: "tech_my_environment"
#   - Only company: "contoso_my_environment"
#   - Neither: "my_environment"
# Primary (UPPERCASE) configuration variables — use these going forward
COMPANY_PREFIX = "healthcare"        # Optional company/customer identifier (e.g., "contoso", "healthcare1")
TECHNICAL_PREFIX = ""  # Optional technical prefix (configurable, default: "msft")


# ⚠️ NOTEBOOK NAMING BEHAVIOR:
# When deploying notebooks, the prefixes are intelligently applied:
# - If notebook source already contains TECHNICAL_PREFIX (e.g., "msft_config_notebook.ipynb"),
#   only COMPANY_PREFIX is added: "healthcare1_msft_config_notebook"
# - If notebook source lacks TECHNICAL_PREFIX (e.g., "my_notebook.ipynb"),
#   both prefixes are added: "healthcare1_msft_my_notebook"
# - If COMPANY_PREFIX is empty, notebooks keep their original names
#
# ⚠️ PIPELINE JSON IMPACT:
# If you add COMPANY_PREFIX, you MUST update pipeline JSON files to reference the new
# prefixed notebook names in:
#   - %run statements in notebooks
#   - Notebook references in pipeline definitions
# Example: Update "%run msft_config_notebook" to "%run healthcare1_msft_config_notebook"


# Deployment options
SAVE_FORMATTED_LOCALLY = True   # Save formatted artifacts to lakehouse
DEPLOY_TO_WORKSPACE = True      # Deploy artifacts to Fabric workspace


print("✓ User configuration loaded")

## 2️⃣ Runtime Auto-Detection

These values are automatically detected from the Fabric runtime context.

In [ ]:
ctx = notebookutils.runtime.context
WORKSPACE_ID = ctx['currentWorkspaceId']
print(f"✓ Workspace ID: {WORKSPACE_ID}")

# Get OneLake endpoint URI (remove https:// prefix)
ENDPOINT_URI = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
print(f"✓ Endpoint URI: {ENDPOINT_URI}")

# Auto-detect workspace name from workspace ID (required for ABFSS paths)
try:
    from sempy.fabric import FabricRestClient
    client = FabricRestClient()
    response = client.get(f"/v1/workspaces/{WORKSPACE_ID}")
    
    if response.status_code == 200:
        workspace_data = json.loads(response.text)
        WORKSPACE_NAME = workspace_data.get("displayName")
        print(f"✓ Workspace Name (auto-detected): {WORKSPACE_NAME}")
    else:
        raise Exception(f"Failed to get workspace info (status {response.status_code})")
        
except Exception as e:
    error_msg = f"""
    ❌ FAILED TO AUTO-DETECT WORKSPACE NAME
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Could not retrieve workspace display name from Fabric API.

    Workspace ID: {WORKSPACE_ID}
    Error: {str(e)}

    The workspace name is required for constructing ABFSS paths to OneLake.
    Please check:
    1. Fabric API connectivity
    2. Workspace permissions
    3. Authentication tokens

    Cannot proceed without workspace name.
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """
    print(error_msg)
    raise Exception(error_msg) from e

print("✓ Runtime context detected")


## 3️⃣ Load Lakehouse Manifest

Load lakehouse configuration from the source of truth: LakehouseHydrationManifest.json

In [ ]:
# Distribution base path - THE primary base for all deployment artifacts
BASE_DIST_PATH = f"abfss://{WORKSPACE_NAME}@{ENDPOINT_URI}/{ARTIFACT_LAKEHOUSE_NAME}.Lakehouse/Files/hds-build-artifacts"

In [ ]:
# =============================================================================
# LAKEHOUSE CONFIGURATION - LOADED FROM MANIFEST
# =============================================================================

def load_lakehouse_manifest() -> Dict[str, Any]:
    """
    Load LakehouseHydrationManifest.json from the distribution folder.
    This is the single source of truth for lakehouse configurations.
    
    Returns:
        Dictionary containing the manifest data
        
    Raises:
        FileNotFoundError: If manifest file is not found
        json.JSONDecodeError: If manifest file is not valid JSON
        Exception: For other errors during loading
    """
    manifest_path = f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}/system-configurations/LakehouseHydrationManifest.json"
    
    print(f"📂 Loading lakehouse manifest from:")
    print(f"   {manifest_path}")
    
    try:
        content = notebookutils.fs.head(manifest_path, 1024 * 1024)  # Read up to 1MB
        manifest_data = json.loads(content)
        
        if not manifest_data or not isinstance(manifest_data, dict):
            raise ValueError(f"Manifest file is empty or not a valid dictionary")
        
        print(f"✅ Successfully loaded manifest with {len(manifest_data)} lakehouse(s)")
        return manifest_data
        
    except FileNotFoundError as e:
        error_msg = f"""
        ❌ MANIFEST FILE NOT FOUND
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        The LakehouseHydrationManifest.json file is required but was not found.

        Expected location: {manifest_path}

        This file is the single source of truth for lakehouse configuration.
        Please ensure:
        1. The distribution artifacts are properly deployed to the lakehouse
        2. The ARTIFACT_VERSION ({ARTIFACT_VERSION}) is correct
        3. The file exists in the healthcare-configuration folder

        Cannot proceed without the manifest file.
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        """
        print(error_msg)
        raise FileNotFoundError(error_msg) from e
        
    except json.JSONDecodeError as e:
        error_msg = f"""
        ❌ MANIFEST FILE IS INVALID JSON
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        The manifest file exists but contains invalid JSON.

        File location: {manifest_path}
        JSON Error: {str(e)}

        Please verify:
        1. The file is valid JSON format
        2. The file was not corrupted during deployment
        3. The file has proper encoding (UTF-8)

        Cannot proceed with corrupted manifest file.
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        """
        print(error_msg)
        raise ValueError(error_msg) from e
        
    except Exception as e:
        error_msg = f"""
        ❌ ERROR LOADING MANIFEST FILE
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        Failed to load the lakehouse manifest file.

        File location: {manifest_path}
        Error type: {type(e).__name__}
        Error message: {str(e)}

        Please check:
        1. File access permissions
        2. Storage connectivity
        3. Lakehouse availability

        Cannot proceed without loading the manifest.
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
        """
        print(error_msg)
        raise Exception(error_msg) from e


def parse_lakehouse_names(manifest: Dict[str, Any]) -> Tuple[Dict[str, str], Dict[str, str]]:
    """
    Extract lakehouse names from manifest and create logical name mappings.
    
    Uses manifest keys directly with minimal normalization (only hyphen→underscore 
    for Python compatibility). No arbitrary transformations.
    
    Examples:
        - "bronze" → logical: "msft_bronze", base: "bronze"
        - "admin" → logical: "msft_admin", base: "admin"
        - "poa-gold" → logical: "msft_poa_gold", base: "poa_gold"
    
    Args:
        manifest: Loaded manifest dictionary
        
    Returns:
        Tuple of:
        - Dictionary mapping logical names to base names (normalized manifest keys)
        - Dictionary mapping base names to original manifest keys (reverse lookup)
        
    Raises:
        ValueError: If manifest is empty or invalid
    """
    if not manifest:
        raise ValueError("Manifest dictionary is empty - cannot parse lakehouse names")
    
    lakehouse_mappings = {}
    base_to_manifest_key = {}
    
    for manifest_key in manifest.keys():
        # Normalize manifest key: hyphen → underscore (for Python identifier compatibility)
        # No other transformations - use manifest keys as-is
        base_name = manifest_key.replace("-", "_")
        
        # Create logical name with technical prefix
        logical_name = f"{TECHNICAL_PREFIX}_{base_name}" if TECHNICAL_PREFIX else base_name
        
        lakehouse_mappings[logical_name] = base_name
        
        # Create reverse mapping: normalized base name → original manifest key
        base_to_manifest_key[base_name] = manifest_key
    
    if not lakehouse_mappings:
        raise ValueError("No valid lakehouse configurations found in manifest")
    
    return lakehouse_mappings, base_to_manifest_key


# Load the manifest file (will raise exception if it fails)
print("📂 Loading lakehouse manifest...")
LAKEHOUSE_MANIFEST = load_lakehouse_manifest()

# Parse lakehouse names from manifest (will raise exception if it fails)
LOGICAL_LAKEHOUSES, BASE_NAME_TO_MANIFEST_KEY = parse_lakehouse_names(LAKEHOUSE_MANIFEST)

# Build lakehouse name mapping for resolution with prefixes applied
# This will be used to resolve actual lakehouse IDs in the workspace
# The mapping is built AFTER build_artifact_name() is defined (in Cell 13)
# For now, we'll use base names and rebuild the map after helper functions load
LAKEHOUSE_NAME_MAP = {k: v for k, v in LOGICAL_LAKEHOUSES.items()}

# Lakehouses to create during setup - use base names from manifest
LAKEHOUSES_TO_CREATE = list(LOGICAL_LAKEHOUSES.values())

print(f"✅ Loaded {len(LOGICAL_LAKEHOUSES)} lakehouses from manifest:")
for logical_name, base_name in LOGICAL_LAKEHOUSES.items():
    print(f"   • {logical_name} → {base_name}")
print(f"✓ Configured {len(LOGICAL_LAKEHOUSES)} logical lakehouses (from manifest)")

# =============================================================================
# MANIFEST HELPER FUNCTIONS
# =============================================================================

def _resolve_manifest_key(lakehouse_name: str) -> str:
    """
    Internal helper: Resolve a lakehouse name to its manifest key.
    
    Tries direct lookup first, then with underscore→hyphen conversion.
    This centralizes the lookup logic used by all manifest helper functions.
    
    Args:
        lakehouse_name: Base name or manifest key
        
    Returns:
        The manifest key
        
    Raises:
        KeyError: If lakehouse not found in manifest
    """
    # Try direct lookup as manifest key first
    if lakehouse_name in LAKEHOUSE_MANIFEST:
        return lakehouse_name
    
    # Try as base name (with underscore → hyphen conversion)
    manifest_key_variant = lakehouse_name.replace("_", "-")
    if manifest_key_variant in LAKEHOUSE_MANIFEST:
        return manifest_key_variant
    
    # Not found - raise error
    available_keys = ", ".join(LAKEHOUSE_MANIFEST.keys())
    raise KeyError(
        f"Lakehouse '{lakehouse_name}' not found in manifest.\n"
        f"Available: {available_keys}"
    )


def get_lakehouse_tables(lakehouse_name: str) -> List[str]:
    """
    Get list of tables for a specific lakehouse from the manifest.
    
    Args:
        lakehouse_name: Base name (e.g., "bronze", "admin", "poa_gold")
                       or manifest key (e.g., "bronze", "admin", "poa-gold")
    
    Returns:
        List of table names defined in the manifest
        
    Raises:
        KeyError: If lakehouse name is not found in manifest
    """
    manifest_key = _resolve_manifest_key(lakehouse_name)
    return LAKEHOUSE_MANIFEST[manifest_key].get("tables", [])


def get_lakehouse_folders(lakehouse_name: str) -> List[Dict[str, Any]]:
    """
    Get folder structure for a specific lakehouse from the manifest.
    
    Args:
        lakehouse_name: Base name (e.g., "bronze", "admin", "poa_gold")
                       or manifest key (e.g., "bronze", "admin", "poa-gold")
    
    Returns:
        List of folder configurations from the manifest
        
    Raises:
        KeyError: If lakehouse name is not found in manifest
    """
    manifest_key = _resolve_manifest_key(lakehouse_name)
    return LAKEHOUSE_MANIFEST[manifest_key].get("folders", [])


def get_lakehouse_source_artifact_path(lakehouse_name: str) -> str:
    """
    Get the source_artifact_path for a specific lakehouse from the manifest.
    This path points to the table definitions directory.
    
    Args:
        lakehouse_name: Base name (e.g., "bronze", "admin", "poa_gold")
                       or manifest key (e.g., "bronze", "admin", "poa-gold")
    
    Returns:
        Path to table definitions
        
    Raises:
        KeyError: If lakehouse name is not found in manifest
        ValueError: If source_artifact_path is not defined in the manifest for this lakehouse
    """
    manifest_key = _resolve_manifest_key(lakehouse_name)
    source_artifact_path = LAKEHOUSE_MANIFEST[manifest_key].get("source_artifact_path")
    
    if not source_artifact_path:
        raise ValueError(
            f"Lakehouse '{manifest_key}' exists in manifest but has no 'source_artifact_path' defined"
        )
    
    return source_artifact_path


def get_manifest_lakehouse_key(base_or_logical_name: str) -> str:
    """
    Convert a base or logical lakehouse name to its manifest key.
    
    This function uses the dynamically generated reverse lookup map,
    eliminating the need for hardcoded special case handling.
    
    Args:
        base_or_logical_name: Base name (e.g., "bronze", "admin", "poa_gold") 
                             or logical name (e.g., "msft_bronze", "msft_admin")
    
    Returns:
        Manifest key (e.g., "bronze", "admin", "poa-gold")
        
    Raises:
        KeyError: If the name cannot be resolved to a manifest key
    
    Examples:
        >>> get_manifest_lakehouse_key("bronze")
        "bronze"
        >>> get_manifest_lakehouse_key("msft_bronze")
        "bronze"
        >>> get_manifest_lakehouse_key("admin")
        "admin"
        >>> get_manifest_lakehouse_key("poa_gold")
        "poa-gold"
    """
    # Check if it's already a manifest key (direct lookup)
    if base_or_logical_name in LAKEHOUSE_MANIFEST:
        return base_or_logical_name
    
    # Check if it's a logical name and get the base name
    if base_or_logical_name in LOGICAL_LAKEHOUSES:
        base_name = LOGICAL_LAKEHOUSES[base_or_logical_name]
    else:
        # Assume it's a base name
        base_name = base_or_logical_name
    
    # Use the reverse lookup map (no hardcoded special cases needed!)
    if base_name in BASE_NAME_TO_MANIFEST_KEY:
        return BASE_NAME_TO_MANIFEST_KEY[base_name]
    
    # Not found - raise error with helpful message
    available_keys = ", ".join(sorted(LAKEHOUSE_MANIFEST.keys()))
    available_base_names = ", ".join(sorted(BASE_NAME_TO_MANIFEST_KEY.keys()))
    raise KeyError(
        f"Cannot resolve '{base_or_logical_name}' to a manifest key.\n"
        f"Available manifest keys: {available_keys}\n"
        f"Available base names: {available_base_names}"
    )


def print_manifest_summary():
    """Print a summary of the lakehouse manifest configuration."""
    print("\n" + "=" * 70)
    print("📊 LAKEHOUSE MANIFEST SUMMARY")
    print("=" * 70)
    
    for manifest_key in sorted(LAKEHOUSE_MANIFEST.keys()):
        config = LAKEHOUSE_MANIFEST[manifest_key]
        tables = config.get("tables", [])
        folders = config.get("folders", [])
        source_artifact_path = config.get("source_artifact_path", "N/A")
        
        print(f"\n🗄️  {manifest_key}")
        print(f"   Tables:    {len(tables)}")
        print(f"   Folders:   {len(folders)}")
        print(f"   Dict Path: {source_artifact_path}")
        
        if tables:
            print(f"   Sample tables: {', '.join(tables[:3])}")
            if len(tables) > 3:
                print(f"                  ... and {len(tables) - 3} more")
    
    print("\n" + "=" * 70)

print("✓ Manifest helper functions loaded")

## 4️⃣ Path Construction

Pre-built paths for artifacts, data, and configurations.

In [ ]:
# =============================================================================
# PATH CONSTRUCTION
# =============================================================================

# Admin lakehouse name - apply prefixes to base name using helper function
# Note: build_artifact_name() is defined later in the Helper Functions section
# So we duplicate the logic temporarily (will be cleaned up when that section executes)
def _build_name(base: str) -> str:
    """Temporary prefix builder (duplicates build_artifact_name logic)."""
    parts = []
    if COMPANY_PREFIX and COMPANY_PREFIX.strip():
        parts.append(COMPANY_PREFIX.strip())
    if TECHNICAL_PREFIX and TECHNICAL_PREFIX.strip():
        parts.append(TECHNICAL_PREFIX.strip())
    parts.append(base)
    return "_".join(parts)

# Apply prefixes to admin lakehouse base name
ADMIN_LAKEHOUSE_NAME = _build_name(ADMIN_DB_NAME)

# Admin lakehouse base path (separate lakehouse for configuration)
ADMIN_LAKEHOUSE_BASE = f"abfss://{WORKSPACE_NAME}@{ENDPOINT_URI}/{ADMIN_LAKEHOUSE_NAME}.Lakehouse"

print("✓ Base paths constructed:")
print(f"  Dist Base:          {BASE_DIST_PATH}")
print(f"  Admin DB Base:      {ADMIN_DB_NAME} (base name)")
print(f"  Admin Lakehouse:    {ADMIN_LAKEHOUSE_NAME} (prefixed)")
print(f"  Admin Lake Base:    {ADMIN_LAKEHOUSE_BASE}")
print(f"\n💡 Individual notebooks construct their specific paths from BASE_DIST_PATH")


## 5️⃣ Notebook Configuration Mappings

Mappings for notebook-to-lakehouse dependencies and formatting rules.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION MAPPINGS
# =============================================================================

# Map notebook filenames to their default lakehouse dependency
NOTEBOOK_LAKEHOUSE_MAPPING = {
    # Config notebook - NO LAKEHOUSE DEPENDENCY
    "msft_config_notebook.ipynb": None,
    
    # Bronze lakehouse notebooks
    "msft_raw_process_movement.ipynb": "bronze",
    "msft_fhir_ndjson_bronze_ingestion.ipynb": "bronze",
    "msft_ahds_fhirservice_export.ipynb": "bronze",
    "msft_imaging_dicom_extract_bronze_ingestion.ipynb": "bronze",
    "msft_imaging_dicom_patch_file_bronze_ingestion.ipynb": "bronze",
    "msft_imaging_dicom_fhir_conversion.ipynb": "bronze",
    "msft_poa_bronze_silver_transformation.ipynb": "bronze",
    "msft_sdoh_raw_extract_bronze_ingestion.ipynb": "bronze",
    "msft_sdoh_bronze_silver_flatten.ipynb": "bronze",
    "msft_claims_extract_bronze_ingestion.ipynb": "bronze",
    "msft_claims_fhir_conversion.ipynb": "bronze",
    "msft_dax_bronze_ingestion.ipynb": "bronze",
    "msft_ai_enrichments_bronze_ingestion.ipynb": "bronze",
    "msft_ai_enrichments_ta4h_execution.ipynb": "bronze",
    "msft_ai_enrichments_medimage_insight_execution.ipynb": "bronze",
    "msft_ai_enrichments_medimage_parse_execution.ipynb": "bronze",
    "msft_ai_enrichments_conversational_data_execution.ipynb": "bronze",
    
    # Silver lakehouse notebooks
    "msft_bronze_silver_flatten.ipynb": "silver",
    "msft_fhir_flattening_sample.ipynb": "silver",
    "msft_generic_dtt_transformation.ipynb": "silver",
    "msft_imaging_dicom_silver_metastore_transformation.ipynb": "silver",
    "msft_dax_silver_ingestion.ipynb": "silver",
    "msft_ai_enrichments_silver_ingestion.ipynb": "silver",
    
    # Gold lakehouses
    "msft_poa_silver_gold_tranformation.ipynb": "poa_gold",
    "msft_omop_silver_gold_transformation.ipynb": "omop",
    "msft_omop_drug_exposure_era_sample.ipynb": "omop",
    "msft_omop_drug_exposure_insights_sample.ipynb": "omop",
    "msft_ci_silver_customerinsights_transformation.ipynb": "customer_insights",
    "msft_cma_silver_gold_transformation.ipynb": "cma_gold",
    
    # Helper notebooks (no lakehouse)
    "msft_alm_helper.ipynb": None,
}

# Notebooks excluded from markdown cell removal during formatting
EXCLUDED_FROM_MARKDOWN_REMOVAL = [
    "msft_ci_silver_customerinsights_transformation.ipynb",
    "msft_poa_silver_gold_tranformation.ipynb",
    "msft_omop_drug_exposure_insights_sample.ipynb",
    "msft_omop_drug_exposure_era_sample.ipynb",
    "msft_fhir_flattening_sample.ipynb",
    "msft_alm_helper.ipynb",
    "msft_generic_dtt_transformation.ipynb",
]

# Notebooks excluded from lakehouse dependency injection
EXCLUDED_FROM_LAKEHOUSE_DEPENDENCY = [
    "msft_config_notebook.ipynb",
    "msft_alm_helper.ipynb",
]

print(f"✓ Notebook mappings configured")
print(f"  Total notebooks: {len(NOTEBOOK_LAKEHOUSE_MAPPING)}")
print(f"  Excluded from markdown removal: {len(EXCLUDED_FROM_MARKDOWN_REMOVAL)}")

## 6️⃣ Table Layer Mapping

Mapping between data layer folders and their target lakehouses.

In [ ]:
# =============================================================================
# TABLE LAYER TO LAKEHOUSE MAPPING (FIXED)
# =============================================================================

FOLDER_TO_LAKEHOUSE_MAP = {}

for manifest_key, config in LAKEHOUSE_MANIFEST.items():
    source_artifact_path = config.get("source_artifact_path")

    if not source_artifact_path:
        continue

    # Extract last folder name
    folder_name = source_artifact_path.rstrip("/").split("/")[-1]

    # Normalize (optional but recommended)
    folder_name = folder_name.lower()

    # Normalize manifest key
    base_name = manifest_key.replace("-", "_")

    # Initialize set (to avoid duplicates)
    if folder_name not in FOLDER_TO_LAKEHOUSE_MAP:
        FOLDER_TO_LAKEHOUSE_MAP[folder_name] = set()

    FOLDER_TO_LAKEHOUSE_MAP[folder_name].add(base_name)

# Convert sets → sorted lists for final output
FOLDER_TO_LAKEHOUSE_MAP = {
    k: sorted(list(v)) for k, v in FOLDER_TO_LAKEHOUSE_MAP.items()
}

# -----------------------------------------------------------------------------
# Debug print
# -----------------------------------------------------------------------------
print(f"✓ Table layer mappings (auto-generated): {len(FOLDER_TO_LAKEHOUSE_MAP)} layers")

for folder, lakehouses in sorted(FOLDER_TO_LAKEHOUSE_MAP.items()):
    lakehouse_str = ", ".join(lakehouses)
    print(f"{folder:20s} → {lakehouse_str}")

## 7️⃣ Helper Functions

Reusable functions for resource resolution.

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

from sempy import fabric
from sempy.fabric import FabricRestClient

# Initialize Fabric REST client (reusable across all notebooks)
FABRIC_CLIENT = FabricRestClient()


def build_artifact_name(base_name: str, use_COMPANY_PREFIX: bool = True, use_TECHNICAL_PREFIX: bool = True) -> str:
    """
    Build artifact name with optional company and technical prefixes.
    
    Naming pattern (underscores only added when prefixes exist):
      - Both prefixes: {COMPANY_PREFIX}_{TECHNICAL_PREFIX}_{base_name}
      - Only technical: {TECHNICAL_PREFIX}_{base_name}
      - Only company: {COMPANY_PREFIX}_{base_name}
      - Neither: {base_name}
    
    Args:
        base_name: Base artifact name (e.g., "my_environment", "msft_bronze")
        use_COMPANY_PREFIX: Include company prefix if available
        use_TECHNICAL_PREFIX: Include technical prefix if available
    
    Returns:
        Prefixed artifact name following the pattern
    
    Examples:
        >>> COMPANY_PREFIX = "contoso"
        >>> TECHNICAL_PREFIX = "msft"
        >>> build_artifact_name("my_env")
        "contoso_msft_my_env"
        >>> build_artifact_name("my_env", use_COMPANY_PREFIX=False)
        "msft_my_env"
        >>> COMPANY_PREFIX = ""
        >>> build_artifact_name("my_env")
        "msft_my_env"
    """
    parts = []
    
    # Add company prefix if enabled and not empty
    if use_COMPANY_PREFIX and COMPANY_PREFIX and COMPANY_PREFIX.strip():
        parts.append(COMPANY_PREFIX.strip())
    
    # Add technical prefix if enabled and not empty
    if use_TECHNICAL_PREFIX and TECHNICAL_PREFIX and TECHNICAL_PREFIX.strip():
        parts.append(TECHNICAL_PREFIX.strip())
    
    # Always add base name
    parts.append(base_name)
    
    # Join with underscores only where needed
    return "_".join(parts)


# Notebooks that should NEVER receive company or technical prefixes
# (critical for %run statements and notebook references that use hardcoded names)
NOTEBOOKS_EXCLUDED_FROM_PREFIXES = [
    "msft_config_notebook"  # Referenced by 50+ notebooks via: %run msft_config_notebook
]


def build_notebook_display_name(source_filename: str, use_COMPANY_PREFIX: bool = True, use_TECHNICAL_PREFIX: bool = True) -> str:
    """
    Build notebook display name with smart prefix handling to avoid duplication.
    
    SPECIAL CASE: Notebooks in NOTEBOOKS_EXCLUDED_FROM_PREFIXES list NEVER receive
    any prefixes, regardless of configuration. This ensures hardcoded %run statements
    continue to work (e.g., %run msft_config_notebook).
    
    If the source filename already contains the technical prefix at the start,
    only add the company prefix. Otherwise, add both prefixes as configured.
    
    Naming pattern:
      - Source already has tech prefix: {COMPANY_PREFIX}_{source_name} (without .ipynb)
      - Source lacks tech prefix: {COMPANY_PREFIX}_{TECHNICAL_PREFIX}_{source_name} (without .ipynb)
      - No company prefix: {source_name} (without .ipynb, keeps existing tech prefix if present)
    
    Args:
        source_filename: Source notebook filename (e.g., "msft_config_notebook.ipynb")
        use_COMPANY_PREFIX: Include company prefix if available
        use_TECHNICAL_PREFIX: Include technical prefix if available (only if not already in source)
    
    Returns:
        Prefixed notebook display name (without .ipynb extension)
    
    Examples:
        >>> COMPANY_PREFIX = "healthcare1"
        >>> TECHNICAL_PREFIX = "msft"
        >>> build_notebook_display_name("msft_config_notebook.ipynb")
        "msft_config_notebook"  # EXCLUDED from prefixes (in NOTEBOOKS_EXCLUDED_FROM_PREFIXES)
        >>> build_notebook_display_name("msft_bronze_silver_flatten.ipynb")
        "healthcare1_msft_bronze_silver_flatten"  # msft already present, added only company
        >>> build_notebook_display_name("my_custom_notebook.ipynb")
        "healthcare1_msft_my_custom_notebook"  # no msft, added both prefixes
        >>> COMPANY_PREFIX = ""
        >>> build_notebook_display_name("my_custom_notebook.ipynb")
        "my_custom_notebook"  # no prefixes to add, kept original
    
    Note:
        ⚠️ PIPELINE IMPACT: If you use prefixed notebook names, update pipeline JSON files
        to reference the new names in %run statements and notebook references.
    """
    # Remove .ipynb extension first
    base_name = source_filename.replace(".ipynb", "")
    
    # Special case: Notebooks in exclusion list NEVER get prefixes (critical for %run statements)
    if base_name in NOTEBOOKS_EXCLUDED_FROM_PREFIXES:
        return base_name
    
    # Check if technical prefix already exists at the start of the filename
    tech_prefix_str = TECHNICAL_PREFIX.strip() if TECHNICAL_PREFIX else ""
    has_tech_prefix = False
    
    if tech_prefix_str and base_name.startswith(f"{tech_prefix_str}_"):
        has_tech_prefix = True
    
    parts = []
    
    # Add company prefix if enabled and not empty
    if use_COMPANY_PREFIX and COMPANY_PREFIX and COMPANY_PREFIX.strip():
        parts.append(COMPANY_PREFIX.strip())
    
    # Add technical prefix only if:
    # 1. It's enabled
    # 2. It's not empty
    # 3. It's NOT already in the source filename
    if use_TECHNICAL_PREFIX and tech_prefix_str and not has_tech_prefix:
        parts.append(tech_prefix_str)
    
    # Always add base name
    parts.append(base_name)
    
    # Join with underscores only where needed
    return "_".join(parts)


def resolve_lakehouse_ids(name_map: Dict[str, str], workspace_id: str = None) -> Dict[str, Optional[str]]:
    """
    Resolve lakehouse artifact IDs in the workspace.
    
    Resolution strategy per entry:
      1. Exact match on Display Name (case-insensitive)
      2. If no exact match, first partial/contains match (case-insensitive)
    
    Args:
        name_map: Dictionary of logical name -> display name
        workspace_id: Workspace ID (defaults to current workspace)
    
    Returns:
        Dictionary of logical name -> lakehouse ID (or None if not found)
    """
    ws_id = workspace_id or WORKSPACE_ID
    items_df = fabric.list_items(workspace=ws_id)
    lh_df = items_df[items_df["Type"] == "Lakehouse"]

    resolved: Dict[str, Optional[str]] = {}
    for key, display_name in name_map.items():
        target = display_name.strip().lower()
        if not target:
            resolved[key] = None
            continue

        # Exact match
        exact = lh_df[lh_df["Display Name"].str.lower() == target]
        if not exact.empty:
            lh_id = exact.iloc[0]["Id"]
            resolved[key] = lh_id
            print(f"  ✓ Lakehouse resolved (exact): {display_name} -> {lh_id}")
            continue

        # Partial/contains match
        partial = lh_df[lh_df["Display Name"].str.lower().str.contains(target)]
        if not partial.empty:
            lh_id = partial.iloc[0]["Id"]
            matched_name = partial.iloc[0]["Display Name"]
            resolved[key] = lh_id
            print(f"  ✓ Lakehouse resolved (partial): {display_name} ~> {matched_name} -> {lh_id}")
            continue

        # No match found
        resolved[key] = None
        print(f"  ⚠ Lakehouse NOT found: '{key}' (expected '{display_name}')")

    return resolved


print("✓ Helper functions loaded")


# =============================================================================
# REBUILD LAKEHOUSE NAME MAP WITH PREFIXES
# =============================================================================
# Now that build_artifact_name() is available, rebuild the lakehouse name map
# to use prefixed display names for proper resolution

LAKEHOUSE_NAME_MAP = {
    logical_key: build_artifact_name(base_name) 
    for logical_key, base_name in LOGICAL_LAKEHOUSES.items()
}

print("✓ Lakehouse name map rebuilt with prefixes:")
for logical, prefixed in LAKEHOUSE_NAME_MAP.items():
    print(f"  {logical:25s} -> {prefixed}")

## 8️⃣ Configuration Summary

Display current configuration for verification.

In [ ]:
# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("📋 DEPLOYMENT CONFIGURATION SUMMARY")
print("=" * 70)

print("\n🎯 Artifact Information:")
print(f"  Version:            {ARTIFACT_VERSION}")
print(f"  Workspace:          {WORKSPACE_NAME}")
print(f"  Lakehouse:          {ARTIFACT_LAKEHOUSE_NAME}")

print("\n🏢 Workspace Context:")
print(f"  Workspace ID:       {WORKSPACE_ID}")

print("\n🏷️ Artifact Naming Prefixes:")
print(f"  Company Prefix:     '{COMPANY_PREFIX}' {'' if COMPANY_PREFIX else '(empty - no company prefix)'}")
print(f"  Technical Prefix:   '{TECHNICAL_PREFIX}' {'' if TECHNICAL_PREFIX else '(empty - no technical prefix)'}")
if COMPANY_PREFIX or TECHNICAL_PREFIX:
    example_name = build_artifact_name("example_artifact")
    print(f"  Example Pattern:    '{example_name}'")
else:
    print(f"  Example Pattern:    'example_artifact' (no prefixes)")

print("\n📓 Notebook Naming (Smart Prefix Handling):")
example_nb1 = build_notebook_display_name("msft_config_notebook.ipynb")
example_nb2 = build_notebook_display_name("my_custom_notebook.ipynb")
print(f"  Source: 'msft_config_notebook.ipynb'")
print(f"    → Deployed as: '{example_nb1}'")
print(f"  Source: 'my_custom_notebook.ipynb'")
print(f"    → Deployed as: '{example_nb2}'")
if COMPANY_PREFIX:
    print(f"  ⚠️  Remember to update pipeline JSONs with new prefixed names!")

print("\n🗄️ Administration Configuration:")
print(f"  Admin DB (base):    {ADMIN_DB_NAME}")
print(f"  Admin LH (prefixed): {ADMIN_LAKEHOUSE_NAME}")
print(f"  Solution Name:      {SOLUTION_NAME} (base name)")
if SOLUTION_NAME == ADMIN_DB_NAME:
    print(f"  ℹ️  Solution name matches admin DB - will use admin lakehouse ID")
print(f"  Pattern:            {COMPANY_PREFIX or '(none)'}{'_' if COMPANY_PREFIX else ''}{TECHNICAL_PREFIX or '(none)'}{'_' if TECHNICAL_PREFIX else ''}{ADMIN_DB_NAME}")

print("\n🌐 Environment:")
print(f"  Target Name:        {TARGET_ENVIRONMENT_NAME or 'Auto-detect'}")

print("\n🗂️ Lakehouses:")
print("  Logical Key              Base Name            Expected Display Name")
print("  " + "-" * 75)
for logical, base in LOGICAL_LAKEHOUSES.items():
    prefixed = build_artifact_name(base)
    print(f"  {logical:25s} {base:20s} {prefixed}")

print("\n📁 Base Paths:")
print(f"  Dist Base:         {BASE_DIST_PATH}...")
print(f"  Admin Lakehouse:   {ADMIN_LAKEHOUSE_BASE}...")

print("\n⚙️ Deployment Options:")
print(f"  Save Locally:       {SAVE_FORMATTED_LOCALLY}")
print(f"  Deploy to Fabric:   {DEPLOY_TO_WORKSPACE}")

print("\n" + "=" * 70)
print("✅ Configuration loaded successfully!")
print("=" * 70)
print("\n💡 Use this config in other notebooks with: %run common_deployment_config\n")


In [ ]:
# -----------------------------------------------------------------------------
# Power BI shared configuration (moved here so deployer and validator share it)
# -----------------------------------------------------------------------------

# Preserve original name builder and install a Power BI-aware override that applies
# prefixes and maps well-known logical display names to short canonical suffixes.

original_build_artifact_name = build_artifact_name
def build_artifact_name(name: str) -> str:
    """Return a canonical, environment-prefixed artifact name (PowerBI-aware)."""
    import re
    # Known mappings for friendly logical PowerBI names -> canonical suffix
    suffix_map = {
        'care management analytics semantic model': 'cma_semantic_model',
        'care management analytics bi report': 'cma_report',
        'patient outreach analytics semantic model': 'poa_semantic_model',
        'patient outreach analytics bi report': 'poa_report',
    }
    def _get_prefix_parts():
        parts = []
        if COMPANY_PREFIX and COMPANY_PREFIX.strip():
            parts.append(str(COMPANY_PREFIX).strip())
        if TECHNICAL_PREFIX and TECHNICAL_PREFIX.strip():
            parts.append(str(TECHNICAL_PREFIX).strip())
        return parts
    def _clean_join(parts):
        joined = "_".join(parts)
        joined = re.sub(r"_+", "_", joined)
        return joined.strip("_")
    def _normalize_display_name_for_fallback(s: str) -> str:
        t = s.strip().lower()
        t = re.sub(r"\s+", "-", t)
        t = re.sub(r"[^a-z0-9\-_]", "", t)
        t = re.sub(r"-+", "-", t)
        return t.strip("-")
    key = name.strip().lower()
    prefix_parts = _get_prefix_parts()
    prefix = _clean_join(prefix_parts) if prefix_parts else ""
    if key in suffix_map:
        suffix = suffix_map[key]
        if prefix:
            return f"{prefix}_{suffix}"
        return suffix
    # Default: fallback normalization (spaces -> '-') optionally prefixed
    fallback = _normalize_display_name_for_fallback(name)
    if prefix:
        return f"{prefix}_{fallback}"
    return fallback

print('✓ Power BI build_artifact_name override installed (prefix-aware)')

# Canonical lists for Power BI deployer/validator
SEMANTIC_MODELS = [
    {
        'name': 'Care Management Analytics Semantic Model',
        'dir': f'{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/care-management-analytics/Datasets/CareManagementAnalytics',
        'lakehouse_binding': 'cma-gold',
        'model_file': 'model.bim',
        'definition_file': 'definition.pbism',
        'placeholders': {
            '%%sql_endpoint%%': 'sqlEndpoint',
            '%%default_lakehouse_name%%': 'lakehouseName',
        },
        'id_var': 'CMA_SM_ID',
    },
    {
        'name': 'Patient Outreach Analytics Semantic Model',
        'dir': f'{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/patient-outreach-analytics-advanced/Datasets/PatientOutreachAdvanced',
        'lakehouse_binding': 'poa-gold',
        'model_file': 'model.bim',
        'definition_file': 'definition.pbidataset',
        'placeholders': {
            '%%sql_endpoint%%': 'sqlEndpoint',
            '%%default_lakehouse_name%%': 'lakehouseName',
        },
        'id_var': 'POA_SM_ID',
    },
]

REPORTS = [
    {
        'name': 'Care Management Analytics BI Report',
        'dir': f'{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/care-management-analytics/Reports/CareManagementAnalytics',
        'semantic_model_ref': 'Care Management Analytics Semantic Model',
        'theme_file': 'CY24SU08.json',
    },
    {
        'name': 'Patient Outreach Analytics BI Report',
        'dir': f'{BASE_DIST_PATH}/healthcare-artifacts/{ARTIFACT_VERSION}/patient-outreach-analytics-advanced/Reports/PatientOutreachAdvanced',
        'semantic_model_ref': 'Patient Outreach Analytics Semantic Model',
        'theme_file': 'CY24SU02.json',
    },
]

print(f'✓ Power BI shared config loaded: {len(SEMANTIC_MODELS)} semantic models, {len(REPORTS)} reports')